# Классификация ботов

### Подход к решению

Для решения этой задачи в качестве модели я использовал CatBoost, потому что он обладает хорошей скоростью, а также показывает лучшие результаты, по сравнению с другими реализациями градиентного бустинга. 

Основной прирост в скоре дал feature engineering, я добавил много признаков, разделил их на следующие блоки:
- **количественные** - для каждой куки были посчитаны число событий, объявлений, категорий, локаций и т.д.
- **временные** - рассчитал интервалы между событиями (медиана, перцентили), всплески активности, возраст куки
- **курсорные** - разброс курсора по экрану, крайние точки по разным осям, размах
- **User Agent** - разбил строку платформы на несколько групп, отдельно выделил скриптовые клиенты
- **популярность объявлений** - для каждого просмотра объявления считается (без утечки будущего), сколько других кук уже смотрели это объявление раньше по времени, взял перцентиль от этого признака

В конце провел тест различных конфигураций CatBoost, лучшей оказалась:
- **iterations=500**
- **learning_rate=0.05**

Финальный ответ делаю усреднением 5 моделей CatBoost по разным сидам, и с разным порядком признаков, для исключения случайности

### Результаты

Baseline из стартового кода давал **P@R0.7 = 0.12**. Моя итоговая модель на валидации выдает **P@R0.7 = 0.87**. 

Наибольший вклад дали признаки по таймингу событий и разбросу курсора

### Ограничения

Максимум, который мне удалось достичь, это 0.87. Дальнейший перебор признаков и гиперпараметров не давал прироста, поэтому остановился на такой конфигурации. На это есть несколько причин:

- Скорее всего это уже достаточно близкое значение к идеальному решению, модель уже использует всю информацию о признаках
- Самой сложной для классификации категорией являются боты на нативном Android приложении. Там нет данных о курсоре, а это один из самых сильных признаков, поэтому модели приходится опираться на другие параметры
- CatBoost чувствителен к выбору сида и порядку столбцов. Я получал различия вплоть до 0.05 в скоре на одних и тех же признаках, просто с другим сидом и порядком колонок. Поэтому я использую усреднение в итоговой модели
- В обучающих данных всего 8% ботов, что является достаточно сильным дисбалансом. Поэтому результат в P@R0.7=0.87 для такой задачи и на таких данных я считаю очень хорошим

### Воспроизводимость

Я использую версию Python 3.12.5. Версии библиотек зафиксированы в requirements.txt

Во всех моделях используется усреднение по сидам, они везде зафиксированы (0-4)

## Загрузка данных

In [669]:
import numpy as np
import pandas as pd

train = pd.read_csv('data/train.csv', parse_dates=['cookie_created_at', 'window_start_ts', 'window_end_ts'])
test = pd.read_csv('data/test.csv', parse_dates=['cookie_created_at', 'window_start_ts', 'window_end_ts'])
events = pd.read_csv('data/events.csv.gz', parse_dates=['event_ts'])

print(train.shape, test.shape, events.shape)
print('доля ботов в train:', train.target.mean().round(4))
train.head()

(11091, 5) (4909, 4) (328905, 14)
доля ботов в train: 0.0811


,cookie_id,cookie_created_at,window_start_ts,window_end_ts,target
0,ck_54a059eb7d3ea68b,2025-11-21 09:30:41,2026-04-06,2026-04-07,0
1,ck_7e4de46eeab82974,2025-09-23 10:10:24,2026-04-06,2026-04-07,0
2,ck_9320229ef6304522,2026-03-04 00:08:02,2026-04-06,2026-04-07,0
3,ck_30ccd25bc1714ed9,2026-04-05 10:52:40,2026-04-06,2026-04-07,0
4,ck_a77c5f05948cdeef,2026-01-01 03:54:35,2026-04-06,2026-04-07,0


In [670]:
events.head()

,cookie_id,event_ts,eid,event_name,platform,user_agent,item_id,item_category,item_location,seller_type,search_query,search_page,pointer_x,pointer_y
0,ck_5efbea1befdefe1b,2026-04-26 09:11:24,200,item_view,desktop,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,1027450.0,elektronika,kaliningrad,pro,NaN,NaN,NaN,NaN
1,ck_c4ca1434f3778f1d,2026-04-20 14:04:31,100,search_results_view,web,Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:1...,NaN,telefony,habarovsk,NaN,iphone 13 128,4.0,NaN,NaN
2,ck_d274382b19488771,2026-04-13 12:02:56,200,item_view,WEB,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,1048743.0,kvartiry_prodazha,kirov,private,NaN,NaN,675.0,276.0
3,ck_fbbed2ff14944ce9,2026-04-08 06:12:14,100,search_results_view,web,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,NaN,bytovaya_tehnika,novosibirsk,NaN,пылесос dyson,2.0,NaN,NaN
4,ck_56cc15c7c634cb9f,2026-04-12 17:16:05,100,search_results_view,WEB,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,NaN,odezhda,sankt-peterburg,NaN,костюм мужской,2.0,NaN,NaN


Объединим таблицы train и test, чтобы удобнее было добавлять новые признаки

In [671]:
# добавил assign чтобы у каждой строки было написано, к какому датасету она относится
data = pd.concat([train.assign(split="train"), test.assign(split="test")], ignore_index=True)

Также обнаружил дубликаты в таблице events, их также удалим

In [672]:
events = events.drop_duplicates()

## Окно наблюдения

Признаки считаем только по событиям внутри окна: `window_start_ts <= event_ts < window_end_ts`.
Это требование из условия, а не рекомендация.

In [673]:
def events_in_window(events, meta):
    ev = events.merge(meta[['cookie_id', 'window_start_ts', 'window_end_ts']], on='cookie_id')
    return ev[(ev.event_ts >= ev.window_start_ts) & (ev.event_ts < ev.window_end_ts)]

ev = events_in_window(events, data).sort_values(['cookie_id', 'event_ts']).reset_index(drop=True)
ev['platform_norm'] = ev.platform.str.lower().replace('iphone', 'ios')

## Добавление признаков

### Baseline

Сначала сделаем baseline, который уже был дан в коде

In [674]:
g = ev.groupby('cookie_id')
features = pd.DataFrame(index=data.cookie_id)
features['n_events'] = g.size()
features['n_items'] = g.item_id.nunique()

Теперь напишем функцию обучения модели, и будем запускать ее после каждого блока добавления новых признаков, для сравнения результатов. В качестве модели будем использовать CatBoost (использую 5 моделей с разными сидами, для того чтобы исключить фактор рандома, и правдвио оценивать все новые признаки)

In [675]:
from catboost import CatBoostClassifier
from metric import precision_at_recall

def evaluate(features, num_cols, cat_cols=(), seeds=range(5), iterations=200, lr=0.1):
    df = train[['cookie_id', 'window_start_ts', 'target']].merge(
        features.reset_index(), on='cookie_id', how='left'
    )
    for c in cat_cols:
        df[c] = df[c].fillna('').astype(str)

    y = df['target'].values
    cols = list(num_cols) + list(cat_cols)

    is_valid = df.window_start_ts.ge('2026-04-17').values

    scores = []
    importances = []
    for seed in seeds:
        model = CatBoostClassifier(
            iterations=iterations,
            learning_rate=lr,
            cat_features=list(cat_cols), 
            random_seed=seed, 
            verbose=False
        )
        model.fit(df.loc[~is_valid, cols], y[~is_valid])
        p = model.predict_proba(df.loc[is_valid, cols])[:, 1]
        scores.append(precision_at_recall(y[is_valid], p))
        importances.append(model.get_feature_importance())

    scores = np.array(scores)
    importance = pd.Series(np.mean(importances, axis=0), index=cols).sort_values(ascending=False)
    
    return scores.mean(), importance

Протестируем baseline

In [676]:
num_cols = ['n_events', 'n_items']
score, importance = evaluate(features=features, num_cols=num_cols)
print('P@R0.7 на валидации (среднее по 5 сидам):', round(score, 4))
importance

P@R0.7 на валидации (среднее по 5 сидам): 0.1201


n_items     51.703084
n_events    48.296916
dtype: float64

### Количественные признаки

Теперь перейдем к добавлению своих признаков.
Первым делом я добавил самые базовые признаки по количеству событий:
- **n_categories** - в скольких категориях была кука
- **cats_per_event** - нормированная n_categories (потому что у активных куки, n_categories будет больше, но это не значит что они боты)
- **n_locations** - в скольких локациях кука находилась
- **max_search_page** - до какой страницы доходит кука
- **n_platform** - количество платформ (привел к одному регистру)
- **n_ua** - количество User Agent
- **n_queries** - количество запросов

In [677]:
features['n_categories'] = g.item_category.nunique()
features['cats_per_event'] = features['n_categories'] / features['n_events']
features['n_locations'] = g.item_location.nunique()
features['max_search_page'] = g.search_page.max()
features['n_platform'] = g.platform_norm.nunique()
features['n_ua'] = g.user_agent.nunique()
features['n_queries'] = g.search_query.nunique()

features['max_search_page'] = features['max_search_page'].fillna(0)

Напишем функцию, которая будет возвращать таблицу сравнения по признакам, между людьми и ботами

In [678]:
def compare_stats(features, cols):
    check = features.join(train.set_index('cookie_id')['target'])
    return check.groupby('target')[cols].mean()

In [679]:
count_cols = ['n_events', 'n_items', 'n_categories', 'cats_per_event', 
        'n_locations', 'max_search_page', 'n_platform', 'n_ua', 'n_queries']

compare_stats(features, count_cols)

,n_events,n_items,n_categories,cats_per_event,n_locations,max_search_page,n_platform,n_ua,n_queries
target,,,,,,,,,
0.0,16.618819,8.917190,2.281201,0.242235,5.881868,4.032673,1.456436,1.135793,3.789835
1.0,29.037820,13.779755,1.787542,0.129081,9.789766,8.916574,1.630701,1.112347,4.758621


Ключевые наблюдения:
- **n_events** у ботов практически в 2 раза больше
- **n_categories** у ботов меньше, значит боты сосредоточены на 1-2 категориях целенаправленно, а не мечутся между ними
- **cats_per_event** также в 2 раза меньше, как и n_categories
- **n_locations** - оказался больше у ботов, что не так очевидно. Боты наоборот используют больше локаций
- **max_search_page** - один из самых показательных признаков, боты в 2 раза глубже ищут объявления, вплоть до дальних страниц выдачи
- **n_platform** и **n_ua** - не показали сильной разницы, что у ботов, что у людей, количество устройств примерно одинаковое

Протестируем признаки, которые мы добавили

In [680]:
score, importance = evaluate(features=features, num_cols=count_cols)
print('P@R0.7 на валидации:', round(score, 4))
importance

P@R0.7 на валидации: 0.1889


n_locations        17.994774
n_categories       16.567067
max_search_page    15.283819
n_items            11.873289
cats_per_event     10.942649
n_events           10.561750
n_queries           7.759971
n_platform          7.646823
n_ua                1.369859
dtype: float64

Получили прирост, но пока слабый, 0.18 по сравнению с 0.12. Значит количество событий не сильно влияет на результат

### Признаки по времени

Теперь добавим признаки по времени. Рассчитаем интервалы между текущим и предыдущим событием внутри каждого куки, и между текущим и следующим событиями. На основе этих признаков создадим новые:
- **median_gap** - медиана интервалов, боты выплоняют действия быстрее людей
- **cv_gap** - коэффициент вариации, показывает на сколько сильно интервалы отклонены относительно своего среднего
- **gap_10**, **gap_90** - 10-й и 90-й перцентили интервалов, поможет увидеть хвосты распределения интервалов
- **gap_min** - минимальный интервал между событиями для каждой куки
- **item_time_median**, **search_time_median** - показывают медиану того, сколько времени кука провела на странице объявления до следующего действия

Также внутри каждой куки для каждого события я рассчитал, сколько событий этой же куки произошло за 60 секунд до текущего event_ts, и потом беру максимум из всех. Таким образом получаем максимальный размер активности за окно наблюдения для каждой куки. На основе этого добавил признак
- **max_events_1min** - показывает максимальное количество событий в скользящем окне 60 секунд для каждой куки

Еще я думаю полезным будет признак по времени жизни куки:
- **age_days** - разница между началом временного окна и созданием куки

И плюсом добавил признаки по времени суток:
- **night_share** - активность ночью, с 2 до 6
- **active_hours** - сколько активных уникальных часов было у куки за окно наблюдения
- **first_hour** - время самого первого события

In [681]:
# diff() вычисляет разницу между предыдущим временем и текущим
# dt.total_seconds() переводит эту разницу в секунды
# Так как мы до этого отсортировали ev по времени event_ts, то все будет правильно работать
ev['gap'] = ev.groupby('cookie_id').event_ts.diff().dt.total_seconds()

# diff(-1) делает то же самое, но считает разницу со следующим временем
ev['next_gap'] = -ev.groupby('cookie_id').event_ts.diff(-1).dt.total_seconds()

g = ev.groupby('cookie_id')

features['median_gap'] = g.gap.median()
features['cv_gap'] = g.gap.std() / g.gap.mean()
features['gap_10'] = g.gap.quantile(0.10)
features['gap_90'] = g.gap.quantile(0.90)
features['gap_min'] = g.gap.min()

# rolling делает скользящее окно заданной длины по времени
# мы для каждой строки считаем количество событий внутри данного куки
# в диапазоне времени (event_ts - 60, event_ts)
roll = ev.set_index('event_ts').groupby('cookie_id').eid.rolling('60s').count()

# rolling выдает двухиндексный датафрейм, мы берем максимум по первому индексу,
# а именно cookie_id, получаем максимальное число активности для каждой куки
features['max_events_1min'] = roll.groupby(level=0).max()

meta = data.set_index('cookie_id')
features['age_days'] = (meta.window_start_ts - meta.cookie_created_at).dt.total_seconds() / 86400

features['item_time_median'] = ev[ev.event_name == 'item_view'].groupby('cookie_id').next_gap.median()
features['search_time_median'] = ev[ev.event_name == 'search_results_view'].groupby('cookie_id').next_gap.median()

hours = ev.event_ts.dt.hour
features['night_share'] = hours.between(2, 6).groupby(ev.cookie_id).mean()
features['active_hours'] = hours.groupby(ev.cookie_id).nunique()
features['first_hour'] = g.event_ts.min().dt.hour

In [682]:
features.info()

<class 'pandas.core.frame.DataFrame'>
Index: 16000 entries, ck_54a059eb7d3ea68b to ck_cc90e219f1ebf390
Data columns (total 21 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   n_events            16000 non-null  int64  
 1   n_items             16000 non-null  int64  
 2   n_categories        16000 non-null  int64  
 3   cats_per_event      16000 non-null  float64
 4   n_locations         16000 non-null  int64  
 5   max_search_page     16000 non-null  float64
 6   n_platform          16000 non-null  int64  
 7   n_ua                16000 non-null  int64  
 8   n_queries           16000 non-null  int64  
 9   median_gap          15641 non-null  float64
 10  cv_gap              15049 non-null  float64
 11  gap_10              15641 non-null  float64
 12  gap_90              15641 non-null  float64
 13  gap_min             15641 non-null  float64
 14  max_events_1min     16000 non-null  float64
 15  age_days            16000 

Есть много столбцов, у которых в некоторых строках значения NaN. Я специально не стал заполнять их 0, потому что это не совсем верно. В данном случае NaN означает невозможность посчитать, а не 0. Поэтому подаем в CatBoost признаки с пропусками, он умеет с ними работать

Во всех остальных блоках признаков я также буду оставлять пропущенные значения, потому что заполнять их 0 будет некорректно

In [683]:
time_cols = ['median_gap', 'cv_gap', 'gap_10', 'gap_90', 'gap_min', 'max_events_1min', 
             'age_days', 'item_time_median', 'search_time_median', 'night_share',
             'active_hours', 'first_hour']

compare_stats(features, time_cols)

,median_gap,cv_gap,gap_10,gap_90,gap_min,max_events_1min,age_days,item_time_median,search_time_median,night_share,active_hours,first_hour
target,,,,,,,,,,,,
0.0,226.309665,2.015828,86.359680,2309.484974,63.101478,2.925530,73.908394,397.922031,450.011341,0.130898,3.132457,12.531005
1.0,101.187919,2.645554,54.661633,1347.918009,43.119687,4.844271,48.726411,190.160755,273.026439,0.197023,3.221357,11.151279


Ключевые выводы по признакам:
- **median_gap** - боты действуют более чем в 2 раза быстрее людей, должен быть важный признак
- **gap_10** - самые быстрые действия ботов быстрее человеческих
- **gap_90** - у людей медианная максимальная пауза около 27 минут, в то время как у ботов в районе 2-х минут, что тоже является очень сильным признаком
- **max_events_1min** - у ботов больше активности за минуту
- **age_days** - время жизни куки у ботов примерно в 1,5 раза меньше, чем у людей
- **item_time_median** - люди проводят на странице вдвое больше времени, действительно читают, а не сканируют ее
- **night_share** - активность ночью не сильно различается, много людей также заходят на сайт ночью

Запустим тест модели с новыми признаками:

In [684]:
score, importance = evaluate(features=features, num_cols=count_cols + time_cols)
print('P@R0.7 на валидации:', round(score, 4))
importance

P@R0.7 на валидации: 0.5454


median_gap            10.581641
n_categories           9.191095
gap_90                 9.175117
n_locations            9.127344
gap_min                5.961831
search_time_median     5.510393
item_time_median       5.280311
n_events               4.974003
max_search_page        4.918505
gap_10                 4.868200
age_days               4.740314
n_items                4.644778
cats_per_event         4.260445
cv_gap                 3.627129
first_hour             3.337448
n_queries              2.576088
max_events_1min        2.526749
n_platform             2.325259
night_share            0.909471
n_ua                   0.873630
active_hours           0.590248
dtype: float64

Добавив временные признаки precision вырос до 0.54, улучшение в 3 раза по сравнению с прошлым блоком.

Если говорить про важность признаков, то самые важные признаки с прошлого шага остались, а именно **n_categories**, **n_locations**, **max_search_page**. Но видно, что временные признаки дали очень сильный вклад, топ признаков по важности как я и предполагал, это **median_gap**, **gap_90**, **gap_min**

### User Agent признаки

Я посмотрел на все уникальные значения в столбце user_agent. Всего их 148. Есть строки, в которых явно прописаны платформы, с которых обычно обращаются боты (например, python-urllib3, curl). Их я выделил в отдельную группу. Также разбил оставшиеся на несколько групп по операционной системе, нативному или браузерному приложению. На основе этих групп добавил следующие столбцы:
- **dominant_ua** - самая частая платформа у куки (без разбивки на группы), может понадобиться модели для отслеживания мелких различий
- **ua_group** - самая частая платформа с разбивкой по группам
- **ua_is_script** - встречался ли у куки скриптовый клиент

In [685]:
import re

def classify_ua(ua):
    ua = str(ua)
    if re.search(r'python-urllib3|python-requests|node-fetch|scrapy|curl/|go-http-client', ua, re.I):
        return 'script'
    if 'HeadlessChrome' in ua:
        return 'linux'
    if ua.startswith('Avito/') and 'Android' in ua:
        return 'app_andoid'
    if ua.startswith('Avito/') and 'iPhone' in ua:
        return 'app_ios'
    if 'Windows' in ua:
        return 'windows'
    if 'Macintosh' in ua:
        return 'mac'
    if 'Android' in ua:
        return 'browser_android'
    if 'iPhone' in ua:
        return 'browser_ios'
    return 'other'

uniq = ev.user_agent.drop_duplicates().to_frame()
uniq['is_scripting'] = uniq.user_agent.apply(lambda x: bool(re.search(r'python-urllib3|python-requests|node-fetch|scrapy|curl/|go-http-client|headlesschrome', x, re.I)))
uniq['ua_group'] = uniq.user_agent.apply(classify_ua)
ev = ev.merge(uniq, on='user_agent', how='left')

g = ev.groupby('cookie_id')
features['dominant_ua'] = g.user_agent.agg(lambda x: x.mode().iat[0])
features['ua_group'] = g.ua_group.agg(lambda x: x.mode().iat[0])
features['ua_is_script'] = g.is_scripting.max()

In [686]:
ua_dummy = pd.get_dummies(features['ua_group'], prefix='ua', dtype=int)
compare_stats(ua_dummy, list(ua_dummy.columns))

,ua_app_andoid,ua_app_ios,ua_browser_android,ua_browser_ios,ua_linux,ua_mac,ua_script,ua_windows
target,,,,,,,,
0.0,0.154435,0.036303,0.268446,0.017955,0.020703,0.121468,0.015306,0.365385
1.0,0.200222,0.018910,0.116796,0.010011,0.083426,0.122358,0.048943,0.399333


Видно, что мобильные браузеры и приложения на IOS, наиболее человечные, в них сильно меньше ботов. Больше всего разница видна на платформах **ua_linux** и **ua_script**, там доля ботов в 4 раза больше чем людей. Значения в Windows и Mac практически не различаются внутри групп, там важны конкретные названия платформ, которые мы передаем в **dominant_ua**. Также много трафика ботов исходит с приложения на Android, хотя кажется, что трафик мобильных приложений должен быть более безопасным

In [687]:
ua_num_cols = ['ua_is_script']
ua_cat_cols = ['dominant_ua', 'ua_group']

score, importance = evaluate(
    features=features, 
    num_cols=count_cols + time_cols + ua_num_cols,
    cat_cols=ua_cat_cols
)

print('P@R0.7 на валидации:', round(score, 4))

P@R0.7 на валидации: 0.6044


Добавление блока признаков по User Agent дало прибавку 0.06 к итогому скору. Кажется немного, но на самом деле это хорошая прибавка, которая не является шумом, и действительно помогает модели различать ботов и людей

### Признаки курсора

Теперь добавим признаки на основе данных о позиции и перемещении курсора. Эти признаки также должны хорошо отличать бота от человека, в первом случае курсор будет двигаться только в определенном месте, а у человека он может двигаться по всему экрану. Добавил следующие признаки:
- **pointer_x_std** - разброс значений курсора по X
- **pointer_y_std** - разброс значений курсора по Y
- **pointer_x_max**, **pointer_x_min**, **pointer_y_max**, **pointer_y_min** - крайние точки курсора
- **pointer_edge** - доля событий, где курсор был на краю экрана, по идее у людей должна быть больше
- **pointer_median_step** - медианное расстояние между последовательными положениями курсора, должен показывать, насколько размашисто двигался курсор
- **item_pointer_rate** - доля item-view событий, у которых есть координаты

In [688]:
p = ev.dropna(subset=['pointer_x', 'pointer_y'])
gp = p.groupby('cookie_id')

features['pointer_x_std'] = gp.pointer_x.std()
features['pointer_y_std'] = gp.pointer_y.std()
features['pointer_x_max'] = gp.pointer_x.max()
features['pointer_x_min'] = gp.pointer_x.min()
features['pointer_y_max'] = gp.pointer_y.max()
features['pointer_y_min'] = gp.pointer_y.min()

edge = (p.pointer_x < 200) | (p.pointer_x > 1400) | (p.pointer_y < 200) | (p.pointer_y > 800)
features['pointer_edge'] = edge.groupby(p.cookie_id).mean()

# np.hypot() считает просто евклидову норму, расстаяние между точками,
# a diff() находит разницу между текущей и предыдущей координатой для каждого события
dist = np.hypot(gp.pointer_x.diff(), gp.pointer_y.diff())
features['pointer_median_step'] = dist.groupby(p.cookie_id).median()

iv = ev[ev.event_name == 'item_view']
features['iv_pointer_rate'] = iv.groupby('cookie_id').pointer_x.apply(lambda s: s.notna().mean())

In [689]:
pointer_cols = ['pointer_x_std', 'pointer_y_std', 'pointer_x_max', 'pointer_x_min',
                'pointer_y_max', 'pointer_y_min', 'pointer_edge', 'pointer_median_step', 'iv_pointer_rate']

compare_stats(features, pointer_cols)

,pointer_x_std,pointer_y_std,pointer_x_max,pointer_x_min,pointer_y_max,pointer_y_min,pointer_edge,pointer_median_step,iv_pointer_rate
target,,,,,,,,,
0.0,541.658616,300.983455,1683.447958,226.182609,946.740448,133.114361,0.652518,771.365987,0.350183
1.0,219.685866,200.563416,1065.972112,327.581673,844.681275,190.235060,0.361389,368.961370,0.266605


Ключевые выводы:
- **pointer_x_std** - разброс значений курсора у людей в 2.5 раза шире по X, должен быть очень полезным признаком
- **pointer_x_max**, **pointer_x_min**, **pointer_y_max**, **pointer_y_min** - все гипотезы подтвердились, боты реже доходят до краев экрана чем люди
- **pointer_edge** - как и ожидалось, курсор людей в 2 раза чаще оказывается на краю экрана
- **pointer_median_step** - размах курсора у людей также в 2 раза больше, чем у ботов

Все признаки должны быть сильными, потому что по каждому есть сильная разница в средних и медианах между ботами и людьми

In [690]:
score, importance = evaluate(
    features=features, 
    num_cols=count_cols + time_cols + ua_num_cols + pointer_cols,
    cat_cols=ua_cat_cols
)
print('P@R0.7 на валидации:', round(score, 4))
importance

P@R0.7 на валидации: 0.7648


median_gap             9.076428
n_locations            7.201472
gap_90                 6.762538
n_categories           6.535735
pointer_x_std          5.992198
gap_min                4.188850
gap_10                 3.796149
pointer_x_max          3.785084
max_search_page        3.578444
dominant_ua            3.449198
item_time_median       3.388914
age_days               3.330548
cats_per_event         3.299346
cv_gap                 3.211422
n_items                3.178016
search_time_median     3.158182
n_events               2.986322
pointer_median_step    2.493602
ua_group               2.373580
first_hour             2.178876
max_events_1min        2.171931
pointer_y_std          1.894560
pointer_y_max          1.821245
pointer_edge           1.802788
pointer_y_min          1.689203
n_queries              1.626094
n_platform             1.617954
pointer_x_min          1.186784
iv_pointer_rate        0.713015
active_hours           0.690866
night_share            0.428400
n_ua    

С новыми признаками precision вырос уже до 0.76, значит блок этих признаков полезнее user agent.

По feature_importance видно, что самыми сильными признаками из добавленных стали **pointer_x_std** и **pointer_x_max**. Это достаточно логично, потому что на предыдущем шаге мы посмотрели, что средние и медианы в этих признаках очень сильно различаются между ботами и людьми

### Признаки по событиям и категориям

Я добавил признаки, которые показывают долю событий куки для конкретных действий:
- **share_contact_phone_show** - доля событий показа телефона продавца
- **share_search_results_view** - доля событий просмотра страницы выдачи

Я попробовал добавить такие признаки для всех типо событий, но большинство из них дали отрицательный эффект, и только эти два действительно принесли пользу

In [691]:
share = pd.crosstab(ev.cookie_id, ev.event_name, normalize='index')
features['share_contact_phone_show'] = share['contact_phone_show']
features['share_search_results_view'] = share['search_results_view']

event_cols = ['share_contact_phone_show', 'share_search_results_view']

In [692]:
score, importance = evaluate(
    features=features, 
    num_cols=count_cols + time_cols + pointer_cols + ua_num_cols + event_cols,
    cat_cols=ua_cat_cols
)

print('P@R0.7 на валидации:', round(score, 4))

P@R0.7 на валидации: 0.7991


И действительно, эти два признака дали прирост в качестве 0.03, поэтому также оставим их для итоговой модели

### Признаки по популярности объявлений

Финальный блок признаков, который я решил добавить. Для начала я посчитал для всех объявлений, какое кол-во кук их просматривало раньше по времени, чем текущая кука, чтобы не было утечки. На основе этого сделал признаки:
- **item_pop_mean** - среднее количество просмотров у всех объявлений, которые просматривала кука
- **item_pop_max** - максимальное количество просмотров среди всех объявлений, которые просматривала кука

Но потом я заметил, что эти признаки дрейфуют между train и test. Чем позже дата, тем больше кук успело накопиться в истории объявления. И если в train среднее ~0.7, то в test это ~1.7. Поэтому на тесте эти признаки будут вести себя неправильно. Исправил это тем, что отранжировал куки в каждом дне по этим признакам, и посчитал ранк для каждой куки (в виде перцентилей, от 0 до 1). Таким образом получил признаки
**pop_mean_dayrank** и **pop_max_dayrank**.

In [693]:
# сначала оставляем из всех событий только просмотры объявлений
# с ненулевым item_id, и сортируем сначала по объявлению, а потом 
# по времени
item_views = (
    ev[(ev.event_name == 'item_view') & ev.item_id.notna()]
    [['cookie_id', 'item_id', 'event_ts']]
    .sort_values(['item_id', 'event_ts'])
)

# эта функция идет по всем объявлениям по порядку, 
# на каждом шаге считает, сколько кук уже просмотрело это объявление
# на момент текущего времени, чтобы не было утечки из будущего
def count_earlier_viewers(group):
    seen_cookies = set()
    counts = []
    for cookie_id in group.cookie_id:
        counts.append(len(seen_cookies))
        seen_cookies.add(cookie_id)
    return pd.Series(counts, index=group.index)

# в итоге в столбце popularity для каждого события записано,
# сколько кук просматривало объявление, которое просматривает текущая кука
item_views['popularity'] = (
    item_views.groupby('item_id', group_keys=False)
    .apply(count_earlier_viewers, include_groups=False)
)

pop_by_cookie = item_views.groupby('cookie_id')['popularity'].agg(
    item_pop_mean='mean', item_pop_max='max'
)
features = features.join(pop_by_cookie)

# для того чтобы не было дрейфа между train и test, сортируем куки внутри
# каждого дня отдельно, и присваиваем каждой ее место от 0 до 1
window_day = data.set_index('cookie_id').window_start_ts
features['pop_mean_dayrank'] = features.item_pop_mean.groupby(window_day).rank(pct=True)
features['pop_max_dayrank'] = features.item_pop_max.groupby(window_day).rank(pct=True)


In [694]:
pop_cols = ['pop_mean_dayrank', 'pop_max_dayrank']
compare_stats(features, pop_cols)

,pop_mean_dayrank,pop_max_dayrank
target,,
0.0,0.481098,0.481248
1.0,0.710838,0.709229


В нашем случае 0.5 - это среднее значение. Значение людей близко к среднему, а боты чаще среднего ходят по популярным объявлениям, поэтому у них ранк 0.7. Посмотрим, как это повлияет на результат

In [695]:
score, importance = evaluate(
    features=features, 
    num_cols=count_cols + time_cols + pointer_cols + ua_num_cols + event_cols + pop_cols,
    cat_cols=ua_cat_cols
)

print('P@R0.7 на валидации:', round(score, 4))
importance

P@R0.7 на валидации: 0.8598


median_gap                   7.485924
n_locations                  6.224732
gap_90                       5.677298
pop_mean_dayrank             5.086064
pointer_x_std                5.021547
n_categories                 4.942591
pointer_x_max                4.462415
max_search_page              3.534555
gap_min                      3.503361
gap_10                       3.229328
age_days                     3.071906
item_time_median             2.998299
cats_per_event               2.993460
cv_gap                       2.869711
dominant_ua                  2.864150
n_events                     2.804335
ua_group                     2.715985
pointer_median_step          2.714947
share_contact_phone_show     2.577330
n_items                      2.559663
search_time_median           2.542079
pop_max_dayrank              2.059917
first_hour                   1.986005
share_search_results_view    1.958071
pointer_y_std                1.869442
max_events_1min              1.718528
pointer_y_ma

Признаки дали сильный прирост в 0.06, а **pop_mean_dayrank** находится в топ 5 признаков по важности. Это финальная конфигурация признаков, которые использует модель. Мы уже получили очень хороший скор, а именно 0.86, теперь можно попробовать изменить конфигурацию CatBoost

### Подбор гиперпараметров модели

Взял 4 конфигурации, изменяя только количество итераций и шаг обучения. Другие гиперпараметры решил не трогать и ограничиться этими

In [696]:
configs = {
    'iter=200, lr=0.1': (200, 0.1),
    'iter=500, lr=0.05': (500, 0.05),
    'iter=900, lr=0.05': (900, 0.05),
    'iter=1500, lr=0.03': (1500, 0.03)
}

final_num = count_cols + time_cols + pointer_cols + ua_num_cols + event_cols + pop_cols
final_cat = ua_cat_cols
final_cols = final_num + final_cat

rows = []
for name, params in configs.items():
    scores = evaluate(features, final_num, final_cat, iterations=params[0], lr=params[1])[0]
    rows.append({'Конфигурация': name, 'mean P@R0.7': scores.mean()})

results = pd.DataFrame(rows).sort_values('mean P@R0.7', ascending=False).reset_index(drop=True)
results

,Конфигурация,mean P@R0.7
0,"iter=500, lr=0.05",0.874040
1,"iter=900, lr=0.05",0.868200
2,"iter=1500, lr=0.03",0.862633
3,"iter=200, lr=0.1",0.859761


По итогам тестирования различных конфигураций, лучшие кол-во итераций и шаг обучения следующие:
- **iterations** = 500
- **learning_rate** = 0.05

Такая конфигурация дает скор 0.874, по сравнению с 0.859, у предыдущей модели, которую мы использовали. Поэтому для предсказания на тесте будем использоваь именно эту конфигурацию

## Сабмит

Для итогового ответа буду усреднять ответы 5 моделей CatBoost с разными сидами, а также для каждой модели буду перемешивать порядок признаков, потому что их расположение тоже достаточно сильно влияет на итоговый результат. Это позволит избавиться от шума, а также получить более стабильную оценку

In [697]:
final_num = count_cols + time_cols + pointer_cols + ua_num_cols + event_cols + pop_cols
final_cat = ua_cat_cols
final_cols = final_num + final_cat

X_train = features.loc[train.cookie_id, final_cols].copy()
X_test = features.loc[test.cookie_id, final_cols].copy()
for c in final_cat:
    X_train[c] = X_train[c].fillna('').astype(str)
    X_test[c] = X_test[c].fillna('').astype(str)

y_train = train.set_index('cookie_id').loc[X_train.index, 'target'].values

test_probas = []
for seed in range(5):
    cols = list(np.random.RandomState(seed).permutation(final_cols))
    model = CatBoostClassifier(
        iterations=500,
        learning_rate=0.05,
        cat_features=final_cat, 
        random_seed=seed, 
        verbose=False
    )
    model.fit(X_train[cols], y_train)
    test_probas.append(model.predict_proba(X_test[cols])[:, 1])

sub = pd.DataFrame({
    'cookie_id': X_test.index,
    'score': np.mean(test_probas, axis=0),
})

assert len(sub) == len(test) and sub.score.between(0, 1).all()
sub.to_csv('submission.csv', index=False)
sub.head()

,cookie_id,score
0,ck_315fb710a0e371e7,0.000438
1,ck_a76ee3b3e3e522fd,0.060075
2,ck_94c9a4d382689e82,0.025097
3,ck_8eaf9509ad9462a0,0.001002
4,ck_9a88a5a989cb5bc6,0.017356
